
# EDA Visual – Cuaderno de Prácticas Integradas / Visual EDA – Integrated Practice Workbook
## SOLUCIONES / SOLUTIONS
Formato limpio, bilingüe. Este cuaderno resuelve los ejercicios del notebook de prácticas.



## 1. Descarga automática de datasets / Automatic dataset download


In [ ]:

# %pip -q install pandas seaborn matplotlib plotly

import os, warnings, pathlib
import pandas as pd

warnings.filterwarnings("ignore")
pathlib.Path("datasets").mkdir(exist_ok=True)

def try_download_csv(name, urls, save_as=None):
    save_as = save_as or f"{name}.csv"
    last_err = None
    for u in urls:
        try:
            df = pd.read_csv(u)
            df.to_csv(f"datasets/{save_as}", index=False)
            print(f"✓ {save_as} — {len(df):,} rows")
            return df
        except Exception as e:
            last_err = e
    print(f"✗ {name} — failed ({last_err})")
    return None

print("Downloading datasets...")

_ = try_download_csv("spotify_2023",[
    "https://raw.githubusercontent.com/yogesh-virat/Spotify-2023/main/spotify-2023.csv",
    "https://raw.githubusercontent.com/abdullahfawzy/Spotify-Dataset-2023/main/spotify-2023.csv",
    "https://raw.githubusercontent.com/raulgomez99/spotify-eda-2023/main/spotify-2023.csv",
], save_as="spotify_2023.csv")

_ = try_download_csv("titanic",[
    "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv",
    "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv",
], save_as="titanic.csv")

_ = try_download_csv("penguins",[
    "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv",
], save_as="penguins.csv")

_ = try_download_csv("flights",[
    "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/flights.csv",
], save_as="flights.csv")

_ = try_download_csv("happiness",[
    "https://raw.githubusercontent.com/plotly/datasets/master/2019.csv",
    "https://raw.githubusercontent.com/ajaykuma/DataSets/master/WorldHappinessReport.csv",
], save_as="happiness.csv")

_ = try_download_csv("netflix_titles",[
    "https://raw.githubusercontent.com/erikgregoryweb/Netflix-Data-EDA/main/netflix_titles.csv",
    "https://raw.githubusercontent.com/vega/vega-datasets/master/data/netflix_titles.csv",
], save_as="netflix_titles.csv")

print("Done.")



### Datasets disponibles / Available datasets


In [ ]:

import os, pandas as pd
print(os.listdir("datasets"))



## 2. NumPy — Operaciones y análisis numérico / NumPy — Operations and numeric analysis


In [ ]:

import numpy as np

# 2.1 Array 4x4 y diagonal
a = np.arange(16).reshape(4,4)
diag = np.diag(a)
print(a)
print("Diagonal:", diag)

# 2.2 1000 valores ~N(0,1): media y desviación
x = np.random.randn(1000)
print("mean=%.3f std=%.3f" % (x.mean(), x.std()))

# 2.3 Cuadrados sin bucles
x = np.array([1,2,3,4,5])
print("squares:", x**2)

# 2.4 Simulación: P(X>4) con un dado justo (1..6)
rolls = np.random.randint(1,7, size=10_000)
p_gt4 = (rolls > 4).mean()
print("P(X>4) ~", p_gt4)



## 3. pandas — Titanic (EDA estructurado) / pandas — Titanic (structured EDA)


In [ ]:

import pandas as pd
titanic = pd.read_csv("datasets/titanic.csv")
titanic.head()


In [ ]:

titanic.info()
titanic.isna().sum()


In [ ]:

# Estadísticos por clase y sexo
titanic.groupby(["Pclass","Sex"])["Survived"].agg(["mean","count"]).reset_index()


In [ ]:

# 'IsChild' y supervivencia por Pclass e IsChild
titanic['IsChild'] = titanic['Age'] < 12
res = titanic.groupby(["Pclass","IsChild"])["Survived"].mean().reset_index()
res



## 4. Seaborn — Spotify 2023 (EDA visual)


In [ ]:

import seaborn as sns, matplotlib.pyplot as plt
sns.set_theme(style="whitegrid", palette="muted")

spotify = pd.read_csv("datasets/spotify_2023.csv")
spotify.columns = [c.strip().lower().replace(' ','_').replace('%','perc').replace('-','_') for c in spotify.columns]

c_pop   = "popularity" if "popularity" in spotify.columns else None
c_dance = "danceability_perc" if "danceability_perc" in spotify.columns else ("danceability" if "danceability" in spotify.columns else None)
c_genre = "genre" if "genre" in spotify.columns else ("track_genre" if "track_genre" in spotify.columns else None)

if c_pop:
    sns.histplot(spotify[c_pop], bins=30, kde=True)
    plt.title("Distribución de popularidad / Popularity distribution")
    plt.show()


In [ ]:

# Popularidad media por género (top 8)
import numpy as np

if c_pop and c_genre:
    top8 = spotify[c_genre].value_counts().head(8).index
    dfg = spotify[spotify[c_genre].isin(top8)]
    sns.barplot(data=dfg, x=c_genre, y=c_pop, estimator=np.mean, errorbar='sd')
    plt.title("Popularidad media por género (top 8)")
    plt.xticks(rotation=30)
    plt.show()


In [ ]:

# Dispersión popularity vs danceability (muestra) + lmplot con hue=genre (si aplicable)
if c_pop and c_dance:
    sample = spotify.sample(min(10000, len(spotify)), random_state=42)
    sns.scatterplot(data=sample, x=c_dance, y=c_pop, alpha=0.3)
    plt.title("Popularity vs Danceability (sample)")
    plt.show()

if c_pop and c_dance and c_genre:
    # limitar a géneros frecuentes para legibilidad
    top10 = spotify[c_genre].value_counts().head(10).index
    dfg2 = spotify[spotify[c_genre].isin(top10)].sample(min(8000, len(spotify)), random_state=42)
    sns.lmplot(data=dfg2, x=c_dance, y=c_pop, hue=c_genre, height=4, aspect=1.4, scatter_kws={'alpha':0.2})
    plt.title("Popularity vs Danceability por género (muestra)")
    plt.show()



## 5. Matplotlib — Penguins (personalización y exportación)


In [ ]:

import matplotlib.pyplot as plt
peng = pd.read_csv("datasets/penguins.csv").dropna()

# Estilo seaborn-like
plt.style.use('seaborn-v0_8-whitegrid')
fig, ax = plt.subplots(figsize=(6,4))
ax.scatter(peng["flipper_length_mm"], peng["body_mass_g"], alpha=0.6, edgecolors="k", linewidths=0.2)
ax.set_title("Penguins: Flipper length vs Body mass")
ax.set_xlabel("Flipper length (mm)")
ax.set_ylabel("Body mass (g)")
ax.annotate("Annotation example",
            xy=(peng["flipper_length_mm"].median(), peng["body_mass_g"].median()),
            xytext=(190, 6000),
            arrowprops=dict(arrowstyle="->", lw=1))
plt.tight_layout()
plt.savefig("penguins_scatter.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:

# Estilo ggplot + línea de referencia
plt.style.use('ggplot')
fig, ax = plt.subplots(figsize=(6,4))
ax.scatter(peng["flipper_length_mm"], peng["body_mass_g"], alpha=0.6, edgecolors="k", linewidths=0.2)
ax.set_title("Penguins: Flipper length vs Body mass (ggplot)")
ax.set_xlabel("Flipper length (mm)")
ax.set_ylabel("Body mass (g)")
ax.axhline(peng["body_mass_g"].mean(), ls="--", lw=1, color="black")
plt.tight_layout()
plt.savefig("penguins_scatter_ggplot.png", dpi=300, bbox_inches="tight")
plt.show()



## 6. Plotly Express — Flights (interactivo)


In [ ]:

import plotly.express as px
flights = pd.read_csv("datasets/flights.csv")
fig = px.line(flights, x="month", y="passengers", color="year",
              title="Passengers by month and year")
fig.show()


In [ ]:

# Heatmap de estacionalidad
pivot = flights.pivot(index="year", columns="month", values="passengers")
fig = px.imshow(pivot, aspect="auto", title="Seasonality heatmap (passengers)")
fig.update_xaxes(title="month")
fig.update_yaxes(title="year")
fig.show()



## 7. Proyecto EDA libre / Open EDA project
Plantilla de inicio para soluciones.


In [ ]:

# Ejemplo de punto de partida (cambiar dataset según preferencia)
# name = "spotify_2023.csv"
# df = pd.read_csv(f"datasets/{name}")
# df.head()



## 8. Explorador de datasets (opcional) / Optional dataset explorer


In [ ]:

import os, pandas as pd
available = os.listdir("datasets")
print("Available:", available)

# name = available[0]
# df = pd.read_csv(f"datasets/{name}")
# df.head()
